# 愛你愛我愛他（LAWA）— 去人聲（Colab）
「LAWA - 〈愛你愛我愛他〉Who You Love (Official MV)」伴奏產出 notebook。流程：yt-dlp 抓音源 → MelBand Roformer（Kim Inst 系列）分離 → 兩段式 loudnorm → FLAC（stereo 44100Hz 24-bit）→ 下載。**使用方式**：執行階段 → 變更執行階段類型 → 選 **T4 GPU** → 全部執行。最後一格會觸發瀏覽器下載 `_伴奏.flac`。

In [ ]:
!pip install -q audio-separator yt-dlp
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '無（請切 T4）')

In [ ]:
# 下載音源（僅音訊，最高可用位元率）
URL = "https://www.youtube.com/watch?v=CbY1FfLeQNQ"
!yt-dlp -f "ba/b" -o "/content/source.%(ext)s" --no-playlist "{URL}"
!ls -la /content/source.*

In [ ]:
# 轉 PCM WAV（44.1kHz stereo）供分離引擎輸入
!ffmpeg -y -i /content/source.* -c:a pcm_s16le -ar 44100 /content/input.wav
!ls -la /content/input.wav

In [ ]:
# 動態選模：優先 MelBand Roformer Kim | Inst V2，退而求其次 V1 Plus 系
import json, os, audio_separator
p = os.path.join(os.path.dirname(audio_separator.__file__), 'models.json')
rl = json.load(open(p, encoding='utf-8'))['roformer_download_list']
pref = ['MelBand Roformer Kim | Inst V2', 'MelBand Roformer Kim | Inst V1 Plus',
        'MelBand Roformer Kim | Inst V1 (E) Plus', 'MelBand Roformer Kim | Inst V1 (E)']
key = next((k for pp in pref for k in rl if pp in k), None)
assert key, '找不到可用的 MelBand Inst 模型'
ckpt = list(rl[key].keys())[0]
print('選用模型 friendly name:', key)
print('ckpt:', ckpt)

In [ ]:
# 分離（Roformer 走 torch+CUDA；輸出 WAV 中間檔）
from audio_separator.separator import Separator
sep = Separator(model_file_dir='/content/models', output_dir='/content/out',
                output_format='WAV', sample_rate=44100)
sep.load_model(model_filename=ckpt)
out = sep.separate(audio_file_path='/content/input.wav')
print('separate() 回傳:', out)
inst = next((f for f in out if 'instrumental' in os.path.basename(f).lower()
             or 'no_vocals' in os.path.basename(f).lower()), None)
assert inst and os.path.getsize(inst) > 0, '伴奏檔缺失或 0 byte'
print('伴奏:', inst)

In [ ]:
# 兩段式 loudnorm → FLAC（stereo / 44100Hz / 24-bit）
# 第一段：量測
!ffmpeg -hide_banner -i "{inst}" -af "loudnorm=I=-14:TP=-1.5:LRA=11:print_format=json" -f null - 2>&1 | tail -20 | tee /content/ln1.txt
import re, json as J
txt = open('/content/ln1.txt', encoding='utf-8', errors='ignore').read()
m = re.search(r'\{[^{}]*"input_i"[^{}]*\}', txt, re.S)
meas = J.loads(m.group(0))
print('量測:', {k: meas[k] for k in ('input_i', 'input_tp', 'input_lra')})
# 第二段：linear 套用（量化後數值）
ii, itp, ilra, ioff = meas['input_i'], meas['input_tp'], meas['input_lra'], meas['input_thresh']
tt, tth = meas['target_i'], meas['target_tp'], meas['target_lra'], meas['target_thresh']
af = f"loudnorm=I=-14:TP=-1.5:LRA=11:measured_I={ii}:measured_TP={itp}:measured_LRA={ilra}:measured_thresh={ioff}:offset={meas['target_offset']}:linear=true"
r = !ffmpeg -y -hide_banner -i "{inst}" -af "{af}" -ar 44100 -c:a flac "/content/LAWA - 愛你愛我愛他_伴奏.flac"
print('rc =', r.returncode if hasattr(r, 'returncode') else 'done')

In [ ]:
# 驗證交付規格：FLAC / stereo / 44100Hz，並量測最終響度
OUT = "/content/LAWA - 愛你愛我愛他_伴奏.flac"
!ffprobe -v error -show_entries stream=codec_name,sample_rate,channels,channel_layout,bits_per_raw_sample -of default=noprint_wrappers=1 "{OUT}"
!ffmpeg -hide_banner -i "{OUT}" -af "loudnorm=print_format=summary" -f null - 2>&1 | tail -15
import os
print('大小:', os.path.getsize(OUT), 'bytes')

In [ ]:
# 下載成品（瀏覽器會跳出下載）
from google.colab import files
files.download(OUT)